# 🇰🇷 Hello Clova Agent — HyperCLOVA X SEED Think-14B

**한국어 프롬프트 한 줄 → Reveal.js 발표 슬라이드 자동 생성**  
**국산 LLM 전용 버전 | LLM: HyperCLOVA X SEED Think-14B (Naver)**

---

## 이 노트북의 목적

Gamma, SkyAI 같은 발표 자동 생성 서비스의 **핵심 파이프라인을 직접 구현하고 체험**합니다.  
공공기관·기업 환경에서 **국산 LLM 사용 요건**을 충족하기 위해 Naver HyperCLOVA X SEED를 사용합니다.

| 개념 | 이 프로젝트에서 | 코드 위치 |
|------|----------------|----------|
| **API 서버** | vLLM이 LLM을 HTTP API로 제공 | 셀 1~2 |
| **앱 서버** | LangGraph 4-노드 에이전트 파이프라인 | `agent/graph.py` |
| **웹 서버** | Gradio가 브라우저 요청을 처리 | `ui/app.py` |

---

## 사전 준비

> **Colab 런타임 설정 필수**: 런타임 → 런타임 유형 변경 → **T4 GPU** (VRAM 15GB) 선택  
> A100 (40GB) 환경이라면 더 빠르게 동작합니다.

| 항목 | 내용 |
|------|------|
| 모델 | `naver-hyperclovax/HyperCLOVA-X-SEED-Think-14B` |
| 양자화 | 4-bit bitsandbytes (bf16 원본 ~28GB → ~8-10GB로 압축) |
| 서빙 | vLLM (OpenAI-compatible API) |
| 소요 시간 | 최초 실행 시 모델 다운로드 약 10~20분 |

---

## 실행 방법

**셀을 위에서 아래로 순서대로 실행하세요.** (단축키: `Shift + Enter`)  
셀 1에서 서버를 백그라운드로 시작하고, 셀 2~3에서 코드를 준비하는 동안 모델이 로딩됩니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 1/5  ▌ vLLM 설치 + HyperCLOVA X API 서버 백그라운드 시작
#
# [개념] API 서버(API Server)란?
#   LLM 모델을 HTTP 엔드포인트로 제공하는 서버입니다.
#   이 서버가 실행되면 다른 프로그램이 LLM을 마치
#   외부 서비스처럼 호출할 수 있습니다.
#
#   엔드포인트: http://localhost:8000/v1/chat/completions
#
# [vLLM vs Ollama]
#   Ollama: 사용 편의성 중심, 공식 레지스트리 모델만 지원
#   vLLM  : HuggingFace 모든 모델 지원, 고성능, 양자화 옵션 풍부
#   → HyperCLOVA X SEED는 Ollama 레지스트리 미등록 → vLLM 사용
#
# [4-bit 양자화란?]
#   bf16 원본 ~28GB → bitsandbytes 4-bit 변환 시 ~8-10GB
#   정밀도를 약간 희생하는 대신 T4(15GB) VRAM에서 구동 가능
# ══════════════════════════════════════════════════════════════════════

import subprocess, os

HCX_MODEL = "naver-hyperclovax/HyperCLOVA-X-SEED-Think-14B"

print("📦 vLLM + bitsandbytes 설치 중... (약 2~3분 소요)")
!pip install vllm bitsandbytes -q

print(f"\n🚀 vLLM API 서버 백그라운드 시작: {HCX_MODEL}")
print("   4-bit 양자화 적용 — VRAM 약 8-10GB 사용")
print("   모델 다운로드+로딩 약 10~20분 소요 (셀 2~3 실행하는 동안 진행됨)")
print()

vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", HCX_MODEL,
        "--host", "0.0.0.0",
        "--port", "8000",
        "--dtype", "bfloat16",
        "--quantization", "bitsandbytes",
        "--load-format", "bitsandbytes",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--trust-remote-code",
        "--served-model-name", HCX_MODEL,
    ],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

os.environ["VLLM_PID"] = str(vllm_proc.pid)
print(f"✅ vLLM 서버 프로세스 시작됨 (PID: {vllm_proc.pid})")
print("   → 셀 2, 3을 실행하면서 서버 로딩을 기다리세요")
print("   → 로그 확인: !tail -f /tmp/vllm.log")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 2/5  ▌ 프로젝트 코드 준비 (git clone / git pull)
#
# 처음 실행: 저장소 전체를 내려받습니다 (git clone)
# 재실행 시: 최신 코드로 업데이트합니다  (git pull)
# ══════════════════════════════════════════════════════════════════════

import os

REPO_URL = "https://github.com/machine-artisan/Hello-Clova-Agent.git"
REPO_DIR = "Hello-Clova-Agent"

if os.path.exists(REPO_DIR):
    print("🔄 최신 코드로 업데이트 중...")
    !git -C {REPO_DIR} pull
else:
    print("📂 프로젝트 다운로드 중...")
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"\n✅ 현재 경로: {os.getcwd()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 3/5  ▌ 의존성 설치 + 환경변수 설정
#
# [개념] 환경변수(Environment Variable)란?
#   프로그램이 실행되는 환경에 주입하는 설정값입니다.
#   LLM_API_BASE, LLM_MODEL 등을 코드 외부에서 주입해
#   모델을 교체할 때 코드를 수정하지 않아도 됩니다.
# ══════════════════════════════════════════════════════════════════════

import os

print("📦 Python 패키지 설치 중... (약 1분 소요)")
!pip install -r requirements.txt -q

# vLLM OpenAI 호환 엔드포인트로 에이전트 연결
HCX_MODEL = "naver-hyperclovax/HyperCLOVA-X-SEED-Think-14B"
os.environ["LLM_API_BASE"] = "http://localhost:8000/v1"
os.environ["LLM_API_KEY"]  = "EMPTY"      # vLLM은 키 불필요
os.environ["LLM_MODEL"]    = HCX_MODEL

print("\n✅ 설치 및 환경변수 설정 완료")
print(f"   LLM_API_BASE = {os.environ['LLM_API_BASE']}")
print(f"   LLM_MODEL    = {os.environ['LLM_MODEL']}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 4/5  ▌ vLLM 서버 준비 완료 대기
#
# 셀 1에서 백그라운드로 시작한 vLLM 서버가 준비될 때까지 기다립니다.
# Think-14B 모델은 HuggingFace에서 다운로드 후 4-bit 변환이 필요해
# 최초 실행 시 10~20분 소요됩니다.
#
# 오래 걸린다면: !tail -20 /tmp/vllm.log 로 진행상황 확인 가능
# ══════════════════════════════════════════════════════════════════════

import time, urllib.request

HEALTH_URL = "http://localhost:8000/health"
MAX_WAIT   = 1200   # 최대 20분
INTERVAL   = 10     # 10초마다 확인

print("⏳ vLLM 서버 준비 대기 중 (최대 20분)")
print("   진행 상황: !tail -20 /tmp/vllm.log")
print()

elapsed = 0
while elapsed < MAX_WAIT:
    try:
        urllib.request.urlopen(HEALTH_URL, timeout=3)
        print(f"\n✅ vLLM 서버 준비 완료! ({elapsed//60}분 {elapsed%60}초 경과)")
        print("   → 셀 5를 실행해 에이전트를 시작하세요")
        break
    except Exception:
        print(f"   [{elapsed:4d}s] 대기 중...", end="\r")
        time.sleep(INTERVAL)
        elapsed += INTERVAL
else:
    print("\n❌ 서버 시작 시간 초과. 로그를 확인하세요:")
    !tail -30 /tmp/vllm.log

## 🏗️ 시스템 아키텍처 — 에이전트가 동작하는 방식

```
┌─────────────────────────────────────────────────────────┐
│                   브라우저 (여러분의 PC)                 │
└───────────────────────┬─────────────────────────────────┘
                        │  HTTP (공개 URL)
┌───────────────────────▼─────────────────────────────────┐
│         🌐 웹서버  —  Gradio  (포트 7860)               │
│         ui/app.py  :  브라우저 ↔ 에이전트 연결          │
└───────────────────────┬─────────────────────────────────┘
                        │  Python 함수 호출
┌───────────────────────▼─────────────────────────────────┐
│         ⚙️  앱서버  —  LangGraph 파이프라인             │
│   Node1 input_parser → Node2 outline_generator          │
│   Node3 slide_writer → Node4 html_renderer              │
└───────────────────────┬─────────────────────────────────┘
                        │  HTTP (OpenAI API 형식)
┌───────────────────────▼─────────────────────────────────┐
│         🤖 API서버  —  vLLM  (포트 8000)                │
│         /v1/chat/completions  :  LLM 추론 제공          │
└───────────────────────┬─────────────────────────────────┘
                        │  GPU (4-bit bitsandbytes)
┌───────────────────────▼─────────────────────────────────┐
│   🇰🇷 LLM  —  HyperCLOVA X SEED Think-14B  (Naver)    │
│         VRAM ~8-10GB  |  Colab T4 (15GB) 가능           │
└─────────────────────────────────────────────────────────┘
```

| 계층 | 기술 | 역할 |
|------|------|------|
| **웹서버** | Gradio | 브라우저의 HTTP 요청 수신 + 응답 |
| **앱서버** | LangGraph | 비즈니스 로직 (에이전트 파이프라인 실행) |
| **API서버** | vLLM | LLM 추론을 REST API로 제공 |
| **LLM** | HyperCLOVA X SEED Think-14B | 실제 텍스트 생성 (국산 LLM) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 5/5  ▌ 에이전트 실행 🚀
#
# 셀 실행 후 출력되는 공개 URL을 브라우저에서 열어주세요.
#   예) Running on public URL: https://xxxx.gradio.live
#
# 중지하려면: 셀 왼쪽의 ■ 버튼 클릭
# ══════════════════════════════════════════════════════════════════════

print("🚀 Hello Clova Agent (HyperCLOVA X SEED Think-14B) 시작 중...")
print("   아래에 공개 URL이 출력되면 브라우저에서 접속하세요 👇")
print()

!python ui/app.py

---

## 🔬 Phase 2 선택 실행 — 임베딩 모델 (RAG 준비)

아래 셀은 **Phase 2 RAG 연동**을 위한 임베딩 모델 설정입니다.  
Phase 1 체험만 원하면 실행하지 않아도 됩니다.

| 항목 | 내용 |
|------|------|
| 모델 | `BAAI/bge-m3` (다국어 임베딩, 한국어 우수) |
| 용도 | 외부 문서를 벡터로 변환하여 RAG 검색에 활용 |
| 크기 | ~570 MB |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# [선택] Phase 2  ▌ 임베딩 모델 준비 (RAG용)
#
# [개념] 임베딩(Embedding)이란?
#   텍스트를 의미를 담은 숫자 벡터로 변환하는 기술입니다.
#   비슷한 의미의 문장은 벡터 공간에서 가까운 위치에 배치되어
#   유사도 검색(RAG의 핵심)이 가능해집니다.
# ══════════════════════════════════════════════════════════════════════

import os

print("📦 임베딩 관련 패키지 설치 중...")
!pip install langchain-huggingface sentence-transformers -q

from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-m3"

if not os.path.exists(EMBED_MODEL):
    print(f"📥 임베딩 모델 다운로드 중: {EMBED_MODEL}")
    model = SentenceTransformer(EMBED_MODEL)
    model.save(EMBED_MODEL)
    print("✅ 다운로드 완료 — 로컬 캐시에 저장됨")
else:
    print(f"✅ 캐시된 모델 사용: {EMBED_MODEL}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True},
)

test_vec = embeddings.embed_query("안녕하세요, 임베딩 테스트입니다.")
print(f"\n✅ 임베딩 모델 준비 완료")
print(f"   벡터 차원: {len(test_vec)}")
print(f"   샘플 벡터 (앞 5개): {[round(v, 4) for v in test_vec[:5]]}")

---

## 📚 더 알아보기

### 코드 구조 탐색

```
Hello-Clova-Agent/
├── agent/
│   ├── state.py              ← 파이프라인이 공유하는 데이터 구조
│   ├── llm.py                ← vLLM API 클라이언트 (OpenAI 호환)
│   ├── graph.py              ← LangGraph 파이프라인 조립
│   └── nodes/
│       ├── input_parser.py   ← Node 1: 입력 분석
│       ├── outline_generator.py  ← Node 2: 목차 생성 (LLM)
│       ├── slide_writer.py   ← Node 3: 내용 작성 (LLM)
│       └── html_renderer.py  ← Node 4: HTML 변환
└── ui/app.py                 ← Gradio 웹 UI
```

### HyperCLOVA X SEED 모델 라인업

| 모델 | 방식 | VRAM | 특징 |
|------|------|------|------|
| **Think-14B** ← 이 노트북 | 4-bit vLLM | ~8-10 GB | T4 가능, 고품질 |
| Instruct-3B | fp16 vLLM | ~8 GB | 빠른 추론 |
| Think-32B | 4-bit vLLM | ~18 GB | 최고 품질, A100 필요 |

---
*Hello Clova Agent · Phase 1 MVP · LangGraph + vLLM + HyperCLOVA X SEED Think-14B + Reveal.js*